In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch import Tensor

from jaxtyping import Float, Int, Bool

## Working with data

In [2]:
train_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

In [11]:
batch_size = 64

train_dataloader = DataLoader(train_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


## Create Models

定义神经网络必须继承nn.Module类。在`__init__`方法中定义网络的结构，在`forward`方法中定义前向传播的过程

为了加速神经网络的计算，可以把model转移到`cuda`中。

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device {device}")

class NeuralNetwork(nn.Module):
    
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.liner_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10), 
        )
    # don't understand input shape and output shape
    def forward(self, x :Float[Tensor, "batch n c h w"]):
        x = self.flatten(x)
        logits = self.liner_relu_stack(x)
        return logits

# move data to device
model = NeuralNetwork().to(device)
print(model)
 

Using device cuda
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (liner_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


## Optimizing the Model Parameters

In [5]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [6]:
def train(dataloader: DataLoader, 
          model: nn.Module, 
          loss_fn: nn.CrossEntropyLoss, 
          optimizer: torch.optim.SGD):
    # 要注意的点：训练前model需要调用train
    model.train()

    size = len(dataloader.dataset)
    # 使用enumerate将会返回一个从0开始的计数器
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch+1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [7]:
def test(dataloader: DataLoader, 
         model: nn.Module, 
         loss_fn: nn.CrossEntropyLoss):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    # before evaluation, you need to call eval() method 
    model.eval()
    
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss = loss_fn(pred, y)
            test_loss += loss
            # 因为softmax的单调性，故可以直接使用argmax而不需要先softmax
            # Single-element tensors: use item() to convert tensor into one value
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    

In [8]:
epoch = 5
for t in range(epoch):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.299988  [   64/60000]
loss: 2.287832  [ 6464/60000]
loss: 2.272066  [12864/60000]
loss: 2.266073  [19264/60000]
loss: 2.239985  [25664/60000]
loss: 2.220130  [32064/60000]
loss: 2.227313  [38464/60000]
loss: 2.197905  [44864/60000]
loss: 2.194901  [51264/60000]
loss: 2.155415  [57664/60000]
Test Error: 
 Accuracy: 35.3%, Avg loss: 2.158919 

Epoch 2
-------------------------------
loss: 2.170799  [   64/60000]
loss: 2.162045  [ 6464/60000]
loss: 2.111835  [12864/60000]
loss: 2.125983  [19264/60000]
loss: 2.064674  [25664/60000]
loss: 2.009821  [32064/60000]
loss: 2.037188  [38464/60000]
loss: 1.966431  [44864/60000]
loss: 1.973782  [51264/60000]
loss: 1.888872  [57664/60000]
Test Error: 
 Accuracy: 53.0%, Avg loss: 1.902332 

Epoch 3
-------------------------------
loss: 1.936969  [   64/60000]
loss: 1.906422  [ 6464/60000]
loss: 1.805608  [12864/60000]
loss: 1.837791  [19264/60000]
loss: 1.716994  [25664/60000]
loss: 1.676915  [32064/600

### 总结

- 使用`to(device)`：
  - 创建好model时
  - 遍历DataLoader，取出数据时

## Saving model

A common way to save a model is to serialize the internal state dictionary (containing the model parameters).

In [9]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


## Loading Models

In [10]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>